In [14]:
# set up: desktop
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import sqlite3
import time
import seaborn as sns

# path set up:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("root on path:", ROOT)

from src.config import DATA_RAW, DATA_PROCESSED
from src import eda

# database connection set up:
DB = '/Users/admin/Desktop/carbon-portfolio-project-v2/data/carbon.db'
con = sqlite3.connect(DB)
con.execute("PRAGMA foreign_keys = ON;")

root on path: /Users/admin/Desktop/carbon-portfolio-project-v2


In [ ]:
# set up: Google Colab
import sys
import sqlite3
import time
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns

In [ ]:

# mount drive (data artifacts live here — never in git)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# clone fresh, or pull if it already exists (so re-running the cell doesn't error)
import os
REPO = "/content/repo"
if os.path.exists(REPO):
    !cd {REPO} && git pull
else:
    !git clone https://github.com/salinela/carbon-portfolio-project-v2.git {REPO}
%cd {REPO}

In [ ]:
# root on path — mirrors desktop's package-style imports
sys.path.insert(0, REPO)
print("root on path:", REPO)

In [ ]:

# live-reload src edits after a git pull without restarting the runtime
%load_ext autoreload
%autoreload 2

from src import eda
from src import feature_engineering as fe

In [ ]:
# DB copied to LOCAL disk (not the Drive FUSE mount) to avoid SQLite locking.
# Needed to read/query it, not just to rebuild — copy once per session.
DRIVE = "/content/drive/MyDrive/carbon_project_v2"
if not os.path.exists("/content/carbon.db"):
    !cp "{DRIVE}/carbon.db" /content/carbon.db

In [ ]:
DB = "/content/carbon.db"
con = sqlite3.connect(DB)
con.execute("PRAGMA foreign_keys = ON;")
print("fe MIN_PERIODS_FRAC:", fe.MIN_PERIODS_FRAC, "| batched:", hasattr(fe, "build_price_features_batched"))

one-time usage

In [13]:
# %% auto-reload edited src modules (so src/*.py edits take effect without kernel restart)
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
# one-time cleanup of the stale broken view
con.execute("DROP VIEW IF EXISTS v_company_emissions;")
con.commit()

# Phase 0: Data Assembly

In [6]:
meta, meta_diag = eda.build_meta(con)
fy,   fy_diag   = eda.build_firm_year(con)

meta_diag, fy_diag

({'n_companies': 8288,
  'by_universe': {'EU': 7988, 'ETS': 300},
  'eligible_n': 5883,
  'sector_nulls_master': 58,
  'sector_nulls_after_backfill': 58,
  'country_nulls': 0,
  'bvd_nulls': 0,
  'coverage_status_counts': {'mapped_loaded': 5883,
   'unmapped_exchange': 1773,
   'mapped_no_data': 523,
   'no_ticker': 109}},
 {'n_rows': 12487,
  'n_companies': 1306,
  'year_range': (2012, 2025),
  'by_source': {'trucost': 9214, 'ets_registry': 3273},
  'revenue_nulls': 573,
  'intensity_nulls': 573})

In [10]:
fy_ids = set(fy["company_id"])
elig   = meta[meta["eligible"] == 1]
mask   = elig.index.isin(fy_ids)
print("eligible:", len(elig))
print("eligible w/ emissions:", int(mask.sum()))
print(elig[mask]["universe"].value_counts().to_dict())

eligible: 5883
eligible w/ emissions: 1259
{'EU': 978, 'ETS': 281}


## Phase 1: Universe Characterization

### Section A: Compute carbon itensity tiers (source registry x 1-digit NACE x year)

In [15]:
# --- cohort flags on meta (enables optional carbon-blind comparison) ---
fy_ids = set(fy["company_id"])
meta["has_emissions_data"] = meta.index.isin(fy_ids).astype(int)
meta["carbon_sample"] = ((meta["eligible"] == 1) &
                         (meta["has_emissions_data"] == 1)).astype(int)

# cross-check against master's has_emissions flag
print("has_emissions agree:",
      (meta["has_emissions"] == meta["has_emissions_data"]).mean())
print("carbon_sample n:", int(meta["carbon_sample"].sum()))

has_emissions agree: 0.9639237451737451
carbon_sample n: 1259


In [16]:
# --- nace1 for tiering: master, backfilled from orbis, first digit ---
nace_full = meta["nace_code"].fillna(meta["orbis_nace_code"])
nace1 = nace_full.astype("string").str.extract(r"(\d)")[0]   # index = company_id

# --- recompute tiers ---
fy, tier_diag = eda.compute_tiers(fy, nace1)
tier_diag

{'tier_counts': {'non_ets_high': 2947,
  'non_ets_low': 2911,
  'non_ets_medium': 2872,
  'ets_high': 1074,
  'ets_low': 1044,
  'ets_medium': 1015,
  <NA>: 573,
  'ets_untiered': 41,
  'non_ets_untiered': 10},
 'nace1_nulls': 0,
 'n_untiered': 51,
 'n_tiered': 11863}

In [33]:
d = meta[meta["has_emissions"] != meta["has_emissions_data"]]
print(len(d))
print(d.groupby(["has_emissions", "has_emissions_data"]).size())
print(d["universe"].value_counts().to_dict())

299
has_emissions  has_emissions_data
0              1                     299
dtype: int64
{'ETS': 299}


### Section B: Monthly Tier-based Portfolio Returns Helper

Main objective: for each month, take every firm currently sitting per tier and average their forward returns (equal-weighted basket)

tier_portfolio_returns produces one such series per tie; "do high-carbon baskets earn different returns than low-carbon ones"

The attach_tier_asof step is what tells each company-month which basket it was in at that date, using the 1-July lag so you're never using an emissions figure before it was public.

In [ ]:
# panel load — desktop: your processed dir; Colab: DRIVE
PANEL_DIR = DATA_PROCESSED            # Colab: PANEL_DIR = DRIVE
features = pd.read_parquet(f"{PANEL_DIR}/features_month_end.parquet")
label    = pd.read_parquet(f"{PANEL_DIR}/label_forward_return.parquet")

panel = features.pivot_table(index=["company_id", "date"],
                             columns="signal_name", values="value")
panel = panel.join(label.set_index(["company_id", "date"])["fwd_ret"])
print("panel:", panel.shape)

panel_t = eda.attach_tier_asof(panel, fy)
print("tier coverage:", round(panel_t["carbon_tier"].notna().mean(), 3))

tret = eda.tier_portfolio_returns(panel_t)
print(tret.shape)
tret.tail()